In [ ]:
import pandas            as pd
import numpy             as np
import seaborn           as sns
import matplotlib.pyplot as plt
import seaborn           as sns
import heapq

from sklearn.model_selection   import train_test_split
from sklearn.model_selection   import cross_val_score
from sklearn.model_selection   import GridSearchCV
from sklearn.preprocessing     import LabelEncoder
from sklearn.preprocessing     import RobustScaler,MinMaxScaler,KBinsDiscretizer
from sklearn.pipeline          import Pipeline
from sklearn.linear_model      import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline          import Pipeline
from xgboost                   import XGBRegressor
from sklearn.metrics           import roc_auc_score, make_scorer
from sklearn                   import metrics
from sklearn.model_selection   import TimeSeriesSplit, GridSearchCV
import os

In [ ]:
def rmsle(y, y_):
    log1 = np.nan_to_num(np.array([np.log(v + 1) for v in y]))
    log2 = np.nan_to_num(np.array([np.log(v + 1) for v in y_]))
    calc = (log1 - log2) ** 2
    return np.sqrt(np.mean(calc))

In [ ]:
def downcast_dtypes(df):
    float_cols = [c for c in df if df[c].dtype == "float64"]
    int_cols = [c for c in df if df[c].dtype in ["int64", "int32"]]
    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols] = df[int_cols].astype(np.int16)
    return df

In [ ]:
def grid_search_cv(X,y,test,param_grid):
    RMSLE = metrics.make_scorer(rmsle, greater_is_better=False)
    print("Parameter grid:\n{}".format(param_grid))
    tscv = TimeSeriesSplit(n_splits=5)
    grid_search = GridSearchCV(XGBRegressor(tree_method='gpu_hist'),param_grid,cv=5,verbose=20,n_jobs=2,scoring=RMSLE)
    grid_search.fit(X,y)

    grid_search_results = pd.DataFrame(grid_search.cv_results_)
    scores = grid_search_results.pivot(index='param_learning_rate', columns='param_n_estimators', values='mean_test_score')
    sns.heatmap(scores, annot=True, fmt="g", cmap='viridis')
    plt.xlabel("n_estimators") 
    plt.ylabel("learning_rate") 
    plt.show()


    grid_search.best_estimator_.fit(X,y)   
    y_test_pred = grid_search.best_estimator_.predict(test)

    score = grid_search.best_score_
    return score, y_test_pred

In [ ]:
holiday = pd.read_csv("../input/store-sales-time-series-forecasting/holidays_events.csv")
oil  = pd.read_csv("../input/store-sales-time-series-forecasting/oil.csv")
stores = pd.read_csv("../input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("../input/store-sales-time-series-forecasting/transactions.csv")
train = pd.read_csv("../input/store-sales-time-series-forecasting/train.csv")
test = pd.read_csv("../input/store-sales-time-series-forecasting/test.csv")
sample_submission = pd.read_csv("../input/store-sales-time-series-forecasting/sample_submission.csv")

In [ ]:
#investigating the files
files = [holiday,oil,stores,transactions,train,test,sample_submission]
titles = ["Holiday","Oil","Stores","Transactions","Train","Test","Sample_submission"]

for file, title in zip(files,titles):
    print("##################")
    print(title)
    print(file.columns)
    #print(file)

In [ ]:
for file, title in zip(files,titles):
    print("##################")
    print(title)
    object_cols = []
    number_cols = []
    for col in file.columns:
        if (file[col].dtype == 'object'):
            object_cols.append(col)
        else:
            number_cols.append(col)
    print("{} Object Columns".format(title))
    print(object_cols)
    print("{} Numerical Columns".format(title))
    print(number_cols)
    #print(file[number_cols])
    for col in object_cols:
        print("Unique values in column {} of {}:".format(col,file))
        print(file[col].value_counts())
        #print("Number of unique values".format(len(file[col].unique())))
        print("******************")

In [ ]:
## combine datasets
train = train.merge(holiday, on = 'date', how='left')
train = train.merge(oil, on = 'date', how='left')
train = train.merge(stores, on = 'store_nbr', how='left')
train = train.merge(transactions, on = ['date', 'store_nbr'], how='left')
print(train.columns)
train = train.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

test = test.merge(holiday, on = 'date', how='left')
test = test.merge(oil, on = 'date', how='left')
test = test.merge(stores, on = 'store_nbr', how='left')
test = test.merge(transactions, on = ['date', 'store_nbr'], how='left')
test = test.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

In [ ]:
train = downcast_dtypes(train)
test  = downcast_dtypes(test)

In [ ]:
train['date_parsed'] = pd.to_datetime(train['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(train['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
test['date_parsed'] = pd.to_datetime(test['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(test['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
train['quarter']    = train['date_parsed'].dt.quarter
train['month']      = train['date_parsed'].dt.month
train['year']       = train['date_parsed'].dt.year
train['dayofyear']  = train['date_parsed'].dt.dayofyear
train['weekofyear'] = train['date_parsed'].dt.isocalendar().week

In [ ]:
test['quarter']    = test['date_parsed'].dt.quarter
test['month']      = test['date_parsed'].dt.month
test['year']       = test['date_parsed'].dt.year
test['dayofyear']  = test['date_parsed'].dt.dayofyear
test['weekofyear'] = test['date_parsed'].dt.isocalendar().week

In [ ]:
train['date_parsed'] = train['date_parsed'].values.astype(float)
test['date_parsed'] = test['date_parsed'].values.astype(float)

In [ ]:
print(train.date_parsed)

In [ ]:
#train.drop(['date_parsed'],axis=1,inplace=True)
#test.drop(['date_parsed'],axis=1,inplace=True)

In [ ]:
print("Train columns:")
print(train.columns)

In [ ]:
train[['store_nbr', 'cluster']] = train[['store_nbr', 'cluster']].astype(str)
test[['store_nbr', 'cluster']] = test[['store_nbr', 'cluster']].astype(str)

In [ ]:
## Item Categories object and number columns
object_cols = []
number_cols = []
for col in train.columns:
    if (train[col].dtype == 'object'):
        object_cols.append(col)
    else:
        number_cols.append(col)
print("Training Object Columns")
print(object_cols)
print("Training Numerical Columns")
print(number_cols)

In [ ]:
agg = train.groupby('date').agg({"sales" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="sales",title = "Average Sales by Date")

agg = train.groupby('date').agg({"transactions" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="transactions",title = "Average Transactions by Date")

In [ ]:
agg = train.groupby(['date', 'store_type']).agg({"sales" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

agg = train.groupby(['date', 'store_type']).agg({"transactions" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

In [ ]:
train.drop(['date'],axis=1,inplace=True)
test.drop(['date'], axis=1,inplace=True)

In [ ]:
y = train['sales']
X = train
X.drop(['sales'],axis=1,inplace=True) 

In [ ]:
train[['holiday_type','locale','locale_name','description']] = train[['holiday_type','locale','locale_name','description']].fillna("Work Day")
test[['holiday_type','locale','locale_name','description']] = test[['holiday_type','locale','locale_name','description']].fillna("Work Day")
train.loc[train['transferred'] == True,['holiday_type','locale','locale_name','description']] = "Work Day"
test.loc[test['transferred'] == True,['holiday_type','locale','locale_name','description']] = "Work Day"

train['transferred'] = train['transferred'].fillna(True)
test['transferred'] = test['transferred'].fillna(True)
train['dcoilwtico'] = train['dcoilwtico'].fillna((train['dcoilwtico'].mean()), inplace=False)
test['dcoilwtico'] = test['dcoilwtico'].fillna((test['dcoilwtico'].mean()), inplace=False)

In [ ]:
missing_data_count = train.isnull().sum()
print("Missing Data in Training Set:")
print(missing_data_count)
sns.heatmap(train.isnull())
plt.show()

In [ ]:
missing_data_count = test.isnull().sum()
print("Missing Data in Test Set:")
print(missing_data_count)
sns.heatmap(test.isnull())
plt.show()

In [ ]:
XandTest = pd.concat([X,test])
s = (X.dtypes == 'object')
object_cols = list(s[s].index)
print("Object Columns")
print(object_cols)
label_encoder = LabelEncoder()
for col in object_cols:
    if (col == 'date_parsed'):
        continue
    XandTest[col] = label_encoder.fit_transform(XandTest[col])
    X[col] = label_encoder.transform(X[col])
    test[col]  = label_encoder.transform(test[col])

In [ ]:
columns  = X.columns
pipe     = Pipeline([("scaler_1", RobustScaler()),("scaler_2", MinMaxScaler())])
XandTest = pd.concat([X,test])
XandTest = pipe.fit(XandTest)
X        = pipe.transform(X)
test     = pipe.transform(test)
X        = pd.DataFrame(X,   columns=columns)
test     = pd.DataFrame(test,columns=columns)

In [ ]:
param_grid = {'n_estimators': [2000,5000,10000],
             'learning_rate': [0.01,0.05,0.1]}
print("Parameter grid:\n{}".format(param_grid))

In [ ]:
X_train = X[:2748913]
y_train = y[:2748913]
X_test  = X[2748913:]
y_test  = y[2748913:]

In [ ]:
RMSLE = metrics.make_scorer(rmsle, greater_is_better=False)
xgb = XGBRegressor(tree_method='gpu_hist')
xgb.fit(X_train,y_train)
y_pred_train = xgb.predict(X_train)
y_pred_test  = xgb.predict(X_test)

In [ ]:
error = rmsle(y_pred_test,y_test)
print(error)

In [ ]:
y_pred_train = pd.Series(y_pred_train)
y_pred_test = pd.Series(y_pred_test,index=range(len(X_train),len(X_train)+len(X_test)))
print(y_pred_test)

In [ ]:
X_train['sales'] = y_train
X_train['sales_pred'] = y_pred_train
X_test['sales'] = y_test
X_test['sales_pred'] = y_pred_test

In [ ]:
agg_train_sales = X_train.groupby('date_parsed').agg({"sales" : "mean"}).reset_index()
agg_train_sales_pred = X_train.groupby('date_parsed').agg({"sales_pred" : "mean"}).reset_index()
agg_test_sales = X_test.groupby('date_parsed').agg({"sales" : "mean"}).reset_index()
agg_test_sales_pred = X_test.groupby('date_parsed').agg({"sales_pred" : "mean"}).reset_index()

In [ ]:
plt.figure()
plt.plot(range(len(agg_train_sales)), agg_train_sales['sales'],color='blue', label="train")
#plt.plot(range(len(agg_train_sales),len(agg_train_sales)+len(agg_test_sales)), agg_test_sales['sales'], '-', color='red',label="test")
plt.plot(range(len(agg_train_sales_pred)), agg_train_sales_pred['sales_pred'], '--', color='green',label="prediction train")
#plt.plot(range(len(agg_train_sales_pred),len(agg_train_sales_pred)+len(agg_test_sales_pred)), agg_test_sales_pred['sales_pred'], '--',color='green',label="prediction test")
plt.legend(loc=(1.01, 0))
plt.xlabel("Date")
plt.ylabel("Sales")
#plt.ylim(0,30000)
plt.show()

In [ ]:
score_y, y_pred = grid_search_cv(X,y,test,param_grid)
print("Score is:", score_y)

In [ ]:
test_set = pd.read_csv("../input/store-sales-time-series-forecasting/test.csv")
output  = pd.DataFrame({'id': test_set.id, 'sales': y_pred})
output.to_csv('my_submission_code.csv',index=False)